In [1]:
# 必要なライブラリのインポート
from torch_geometric.datasets import Planetoid

# torch_geometric の Dataset としてダウンロード
dataset = Planetoid(root="./dataset", name="Cora", split="full")

Processing...
Done!


In [2]:
from torch_geometric.transforms import RandomNodeSplit

# ノードを学習データとテストデータに分割
node_splitter = RandomNodeSplit(
    split="train_rest",  # 分割方法
    num_splits=1,  # 分割数
    num_val=0.0,  # 検証データの割合
    num_test=0.4,  # テストデータの割合
    key="y",  # 正解データの属性名
)
splitted_data = node_splitter(dataset._data)
print(splitted_data.node_attrs())
print(splitted_data.train_mask)
print(splitted_data.test_mask)

['test_mask', 'y', 'val_mask', 'x', 'train_mask']
tensor([False, False,  True,  ...,  True,  True, False])
tensor([ True,  True, False,  ..., False, False,  True])


In [3]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, Sequential


# GCNモデルの定義
class GCN(torch.nn.Module):
    def __init__(
        self,
        num_node_features: int,  # 入力層の次元数 = ノードの特徴量の次元数
        projection_dim: int,  # 中間層の次元数
        num_classes: int,  # 出力層の次元数 = 分類先のクラス数
    ) -> None:
        super().__init__()
        self.conv1 = GCNConv(num_node_features, projection_dim)
        self.conv2 = GCNConv(projection_dim, num_classes)

    def forward(self, data: Data) -> torch.Tensor:
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

In [4]:
# GCNモデルのインスタンス化
device = "cuda" if torch.cuda.is_available() else "cpu"
gcn_model = GCN(
    num_node_features=dataset.num_node_features,
    projection_dim=64,
    num_classes=dataset.num_classes,
).to(device)

# 最適化アルゴリズムの選択
gcn_optimizer = torch.optim.Adam(list(gcn_model.parameters()), lr=0.01)